# Compare MoE vs MLP Across All Metrics

This notebook compares one MoE checkpoint against one MLP checkpoint on the same test set, using the same evaluator pipeline as the project code.

It reports:
- full evaluator metrics side by side
- absolute and relative differences
- per-instance win rates for objective, merit, and violations

In [1]:
import os
import glob
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from utils.trainer import load_instance, DEVICE
from utils.evaluator import Evaluator
from eval import load_model_from_checkpoint, resolve_checkpoints, load_single_model
from models.neural_networks import EnsembleMLP

print(f'Device: {DEVICE}')

Device: cpu


## 1) Configure Paths
Set either `*_RUN_DIR` or `*_CKPT_PATH` for each model.

In [2]:
# Optional run directories (auto-resolve model.pt or members/member_*.pt)
MLP_RUN_DIR = None
MOE_RUN_DIR = None
ENS_RUN_DIR = None

# Optional explicit checkpoint paths
MLP_CKPT_PATH = None
MOE_CKPT_PATH = None
ENS_CKPT_PATH = None

# Example:
# MLP_RUN_DIR = 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260316-135658_FSNet_seed0_e300_lr7e-05_n7000' # 3M
MLP_RUN_DIR = 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-181820_FSNet_seed0_e300_lr1e-04_n7000' # 16M
MOE_RUN_DIR = 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172157_FSNet_seed1_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05' # 3M
MOE_RUN_DIR = 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260320-135248_FSNet_seed0_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05' #13M
ENS_RUN_DIR = 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0_e300_lr1e-04_n7000_ens5_vanilla_pre' # 16M

BATCH_SIZE = 512
FORCE_EVAL_MODE = True  # ensure post-processing is active for FSNet/S3Net

In [3]:
def resolve_checkpoint_list(run_dir, ckpt_path, label):
    if ckpt_path:
        if not os.path.isfile(ckpt_path):
            raise FileNotFoundError(f'{label} checkpoint not found: {ckpt_path}')
        return [ckpt_path]

    if run_dir:
        ckpts = resolve_checkpoints(run_dir)
        if len(ckpts) == 0:
            raise ValueError(f'{label}: no checkpoints found under {run_dir}')
        return ckpts

    raise ValueError(f'Set either {label}_RUN_DIR or {label}_CKPT_PATH')


def safe_ratio_pct(delta, baseline):
    denom = abs(baseline) if abs(baseline) > 1e-12 else np.nan
    return 100.0 * delta / denom


def load_model_from_checkpoint_list(ckpt_list, opt_problem):
    if len(ckpt_list) == 1:
        model, cfg = load_model_from_checkpoint(ckpt_list[0], opt_problem)
        return model, cfg

    members = []
    base_cfg = None
    for i, ckpt_path in enumerate(ckpt_list):
        m, cfg = load_single_model(ckpt_path, opt_problem)
        if i == 0:
            base_cfg = cfg
        else:
            for k in ['prob_type', 'prob_name', 'method']:
                if cfg.get(k) != base_cfg.get(k):
                    raise ValueError(f'Ensemble checkpoint mismatch for {k}: {cfg.get(k)} vs {base_cfg.get(k)}')
        members.append(m)
    model = EnsembleMLP(members).to(DEVICE)
    model.eval()
    return model, base_cfg


def collect_per_instance_metrics(model, evaluator, opt_problem, loader, penalty=1e5, vio_tol=1e-5):
    obj_all = []
    merit_l1_all = []
    merit_l2_all = []
    eq_l1_all = []
    eq_l2_all = []
    ineq_l1_all = []
    ineq_l2_all = []

    # For violation-rate reporting:
    # 1) compute per-instance violation-rate (fraction of violated constraints),
    # 2) take batch mean,
    # 3) compare means across batches.
    eq_viol_rate_batch_means = []
    ineq_viol_rate_batch_means = []
    any_viol_rate_batch_means = []

    opt_gap_all = []

    with torch.no_grad():
        for x_batch, y_true in loader:
            x_batch = x_batch.to(DEVICE)
            y_true = y_true.to(DEVICE)

            # IMPORTANT: use evaluator's final-prediction path so ensembles are
            # evaluated with ensemble_post/ensemble_agg semantics.
            y_final = evaluator._get_final_prediction(model, x_batch)

            obj = opt_problem.obj_fn(y_final)
            obj_true = opt_problem.obj_fn(y_true)

            eq_resid = opt_problem.eq_resid(x_batch, y_final)
            ineq_resid = opt_problem.ineq_resid(x_batch, y_final)

            eq_l1 = eq_resid.abs().sum(dim=1)
            eq_l2 = eq_resid.pow(2).sum(dim=1).sqrt()
            ineq_l1 = ineq_resid.abs().sum(dim=1)
            ineq_l2 = ineq_resid.pow(2).sum(dim=1).sqrt()

            merit_l1 = obj + penalty * (eq_l1 + ineq_l1)
            merit_l2 = obj + penalty * (eq_l2 + ineq_l2)
            opt_gap = 100.0 * (obj - obj_true) / obj_true.abs().clamp_min(1e-12)

            eq_vio_mask = eq_resid.abs() > vio_tol
            ineq_vio_mask = ineq_resid.abs() > vio_tol
            all_vio_mask = torch.cat([eq_vio_mask, ineq_vio_mask], dim=1)

            eq_vio_rate_inst = eq_vio_mask.to(torch.float32).mean(dim=1) * 100.0
            ineq_vio_rate_inst = ineq_vio_mask.to(torch.float32).mean(dim=1) * 100.0
            any_vio_rate_inst = all_vio_mask.to(torch.float32).mean(dim=1) * 100.0

            eq_viol_rate_batch_means.append(eq_vio_rate_inst.mean().item())
            ineq_viol_rate_batch_means.append(ineq_vio_rate_inst.mean().item())
            any_viol_rate_batch_means.append(any_vio_rate_inst.mean().item())

            obj_all.append(obj.cpu().numpy())
            merit_l1_all.append(merit_l1.cpu().numpy())
            merit_l2_all.append(merit_l2.cpu().numpy())
            eq_l1_all.append(eq_l1.cpu().numpy())
            eq_l2_all.append(eq_l2.cpu().numpy())
            ineq_l1_all.append(ineq_l1.cpu().numpy())
            ineq_l2_all.append(ineq_l2.cpu().numpy())
            opt_gap_all.append(opt_gap.cpu().numpy())

    return {
        'objective': np.concatenate(obj_all),
        'merit_l1': np.concatenate(merit_l1_all),
        'merit_l2': np.concatenate(merit_l2_all),
        'eq_violation_l1': np.concatenate(eq_l1_all),
        'eq_violation_l2': np.concatenate(eq_l2_all),
        'ineq_violation_l1': np.concatenate(ineq_l1_all),
        'ineq_violation_l2': np.concatenate(ineq_l2_all),
        # These arrays are per-batch means of per-instance violation rates.
        'eq_violation_rate': np.asarray(eq_viol_rate_batch_means, dtype=float),
        'ineq_violation_rate': np.asarray(ineq_viol_rate_batch_means, dtype=float),
        'any_violation_rate': np.asarray(any_viol_rate_batch_means, dtype=float),
        'opt_gap_pct': np.concatenate(opt_gap_all),
    }

In [4]:
mlp_ckpts = resolve_checkpoint_list(MLP_RUN_DIR, MLP_CKPT_PATH, 'MLP')
moe_ckpts = resolve_checkpoint_list(MOE_RUN_DIR, MOE_CKPT_PATH, 'MoE')
ens_ckpts = resolve_checkpoint_list(ENS_RUN_DIR, ENS_CKPT_PATH, 'ENS')

print('MLP checkpoints:', mlp_ckpts)
print('MoE checkpoints:', moe_ckpts)
print('ENS checkpoints:', ens_ckpts)

mlp_raw = torch.load(mlp_ckpts[0], map_location='cpu', weights_only=False)
moe_raw = torch.load(moe_ckpts[0], map_location='cpu', weights_only=False)
ens_raw = torch.load(ens_ckpts[0], map_location='cpu', weights_only=False)

mlp_cfg = copy.deepcopy(mlp_raw['config'])
moe_cfg = copy.deepcopy(moe_raw['config'])
ens_cfg = copy.deepcopy(ens_raw['config'])

required_same = ['prob_type', 'prob_name', 'prob_size', 'method']
for k in required_same:
    if mlp_cfg.get(k) != moe_cfg.get(k):
        raise ValueError(f'Config mismatch for {k}: MLP={mlp_cfg.get(k)} vs MoE={moe_cfg.get(k)}')
    if mlp_cfg.get(k) != ens_cfg.get(k):
        raise ValueError(f'Config mismatch for {k}: MLP={mlp_cfg.get(k)} vs ENS={ens_cfg.get(k)}')

print('Common setup looks compatible.')
print('Method:', mlp_cfg['method'])
print('Problem:', mlp_cfg['prob_type'], '/', mlp_cfg['prob_name'])

MLP checkpoints: ['results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-181820_FSNet_seed0_e300_lr1e-04_n7000/model.pt']
MoE checkpoints: ['results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260320-135248_FSNet_seed0_e300_lr1e-04_n7000_moe4k2_temp1.0_noise0.05/model.pt']
ENS checkpoints: ['results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0_e300_lr1e-04_n7000_ens5_vanilla_pre/members/member_0.pt', 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0_e300_lr1e-04_n7000_ens5_vanilla_pre/members/member_1.pt', 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0_e300_lr1e-04_n7000_ens5_vanilla_pre/members/member_2.pt', 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0_e300_lr1e-04_n7000_ens5_vanilla_pre/members/member_3.pt', 'results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260315-172824_FSNet_seed0

In [5]:
ens_cfg

{'seed': 0,
 'train_size': 7000,
 'val_size': 1000,
 'test_size': 2000,
 'batch_size': 512,
 'test_batch_sizes': [256, 512],
 'eval_step': 10,
 'network': 'MLP',
 'hidden_dim': 1024,
 'num_layers': 4,
 'dropout': 0.1,
 'MoE': {'num_experts': 4,
  'top_k': 2,
  'aux_loss_weight': 0.01,
  'gate_temperature': 1.0,
  'gate_noise_std': 0.0,
  'warmup_epochs': 30,
  'start_temp': 2.0,
  'final_temp': 1.0,
  'gate_noise_final': 0.0,
  'temp_decay_epochs': 200},
 'penalty': {'lr': 0.0001,
  'num_epochs': 1000,
  'obj_weight': 1.0,
  'eq_pen_weight': 10.0,
  'ineq_pen_weight': 10.0,
  'dist_weight': 5.0,
  'scale': 1000,
  'val_tol': 1e-07,
  'test_val_tol': 1e-09,
  'decay_tol_step': 100,
  'memory_size': 30,
  'max_iter': 50,
  'max_diff_iter': 30},
 'adaptive_penalty': {'lr': 0.0001,
  'num_epochs': 1000,
  'obj_weight': 1.0,
  'eq_pen_weight': 10.0,
  'ineq_pen_weight': 10.0,
  'dist_weight': 5.0,
  'scale': 1000,
  'val_tol': 1e-07,
  'test_val_tol': 1e-09,
  'decay_tol_step': 100,
  'memo

In [6]:
# Build one shared optimization problem instance
base_cfg = copy.deepcopy(mlp_cfg)
if FORCE_EVAL_MODE:
    base_cfg['_eval_only'] = True

opt_problem, _ = load_instance(base_cfg)
test_loader = DataLoader(opt_problem.test_dataset, batch_size=BATCH_SIZE, shuffle=False)

mlp_model, _ = load_model_from_checkpoint_list(mlp_ckpts, opt_problem)
moe_model, _ = load_model_from_checkpoint_list(moe_ckpts, opt_problem)
ens_model, _ = load_model_from_checkpoint_list(ens_ckpts, opt_problem)

mlp_eval_cfg = copy.deepcopy(mlp_cfg)
moe_eval_cfg = copy.deepcopy(moe_cfg)
ens_eval_cfg = copy.deepcopy(ens_cfg)
if FORCE_EVAL_MODE:
    mlp_eval_cfg['_eval_only'] = True
    moe_eval_cfg['_eval_only'] = True
    ens_eval_cfg['_eval_only'] = True

# Explicitly align ensemble behavior with test_ensemble-style settings.
ens_eval_cfg['ensemble_post'] = 'post'
ens_eval_cfg['ensemble_agg'] = 'best_merit'

# Enable MoE candidate-level post-processing/selection.
moe_eval_cfg['moe_post'] = 'post'
moe_eval_cfg['moe_agg'] = 'best_merit'

mlp_evaluator = Evaluator(opt_problem, mlp_eval_cfg['method'], mlp_eval_cfg)
moe_evaluator = Evaluator(opt_problem, moe_eval_cfg['method'], moe_eval_cfg)
ens_evaluator = Evaluator(opt_problem, ens_eval_cfg['method'], ens_eval_cfg)

print(f'Test size: {len(opt_problem.test_dataset)}  | batch size: {BATCH_SIZE}')
print(f'Ensemble member count (ENS): {len(ens_ckpts)}')
print(f"MoE eval mode: moe_post={moe_eval_cfg.get('moe_post')}  moe_agg={moe_eval_cfg.get('moe_agg')}")
print(f"ENS eval mode: ensemble_post={ens_eval_cfg.get('ensemble_post')}  ensemble_agg={ens_eval_cfg.get('ensemble_agg')}")

Test size: 2000  | batch size: 512
Ensemble member count (ENS): 5
MoE eval mode: moe_post=post  moe_agg=best_merit
ENS eval mode: ensemble_post=post  ensemble_agg=best_merit


In [7]:
# Model size summary (parameters and approximate memory)
def model_size_report(model, name):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    total_mb = total_bytes / (1024 ** 2)

    print(f'[{name}]')
    print(f'  total params:      {total_params:,}')
    print(f'  trainable params:  {trainable_params:,}')
    print(f'  param memory:      {total_mb:.2f} MiB')
    print('')

model_size_report(mlp_model, 'MLP')
model_size_report(moe_model, 'MoE')
model_size_report(ens_model, 'ENS')

[MLP]
  total params:      12,898,404
  trainable params:  12,898,404
  param memory:      98.41 MiB

[MoE]
  total params:      13,214,300
  trainable params:  13,214,300
  param memory:      100.82 MiB

[ENS]
  total params:      16,517,620
  trainable params:  16,517,620
  param memory:      126.02 MiB



In [8]:
# Aggregate evaluator metrics: MLP vs MoE vs ENS
mlp_metrics = mlp_evaluator.evaluate(mlp_model, test_loader, split_name='MLP')
moe_metrics = moe_evaluator.evaluate(moe_model, test_loader, split_name='MoE')
ens_metrics = ens_evaluator.evaluate(ens_model, test_loader, split_name='ENS')

metrics_by_model = {
    'MLP': mlp_metrics,
    'MoE': moe_metrics,
    'ENS': ens_metrics,
}

all_keys = sorted(set().union(*[set(v.keys()) for v in metrics_by_model.values()]))
rows = []
for metric in all_keys:
    mlp_val = float(metrics_by_model['MLP'].get(metric, np.nan))
    moe_val = float(metrics_by_model['MoE'].get(metric, np.nan))
    ens_val = float(metrics_by_model['ENS'].get(metric, np.nan))
    rows.append({
        'metric': metric,
        'MLP': mlp_val,
        'MoE': moe_val,
        'ENS': ens_val,
        # 'MoE_minus_MLP': moe_val - mlp_val,
        # 'ENS_minus_MLP': ens_val - mlp_val,
        # 'ENS_minus_MoE': ens_val - moe_val,
    })

cmp_df = pd.DataFrame(rows).sort_values('metric').reset_index(drop=True)
pd.set_option('display.max_rows', 500)
cmp_df

,metric,MLP,MoE,ENS
0,avg_inference_time,6.622137e-01,1.093931e+00,2.548217e+00
1,eq_violation_l1_max,1.096345e-03,9.272472e-04,8.918865e-05
2,eq_violation_l1_mean,5.350399e-05,3.858921e-05,1.767688e-05
3,eq_violation_l2_max,3.884471e-08,3.333758e-08,2.566218e-10
4,eq_violation_l2_mean,4.273989e-10,1.837687e-10,1.452828e-11
5,eq_violation_max_max,6.990406e-05,5.698765e-05,5.178108e-06
6,eq_violation_max_mean,3.292425e-06,2.422518e-06,1.105775e-06
7,ineq_violation_l1_max,2.281083e-05,1.198702e-05,1.647024e-06
8,ineq_violation_l1_mean,2.405282e-07,1.309152e-07,2.402539e-08
9,ineq_violation_l2_max,4.854910e-10,1.942373e-10,2.779738e-12


In [9]:
# Per-instance comparison: MLP vs MoE vs ENS
mlp_inst = collect_per_instance_metrics(mlp_model, mlp_evaluator, opt_problem, test_loader)
moe_inst = collect_per_instance_metrics(moe_model, moe_evaluator, opt_problem, test_loader)
ens_inst = collect_per_instance_metrics(ens_model, ens_evaluator, opt_problem, test_loader)

summary_rows = []
# for metric_name in [
#     'objective', 'merit_l1', 'eq_violation_l1', 'ineq_violation_l1', 'opt_gap_pct',
#     'merit_l2', 'eq_violation_l2', 'ineq_violation_l2',
#     'eq_violation_rate', 'ineq_violation_rate', 'any_violation_rate',
# ]:
for metric_name in [
    'opt_gap_pct', 'eq_violation_l1', 'ineq_violation_l1', 'merit_l1',
    'eq_violation_rate', 'ineq_violation_rate', 'eq_violation_l2', 'ineq_violation_l2', 'merit_l2',
]:
    mlp_vals = mlp_inst[metric_name]
    moe_vals = moe_inst[metric_name]
    ens_vals = ens_inst[metric_name]

    stacked = np.stack([mlp_vals, moe_vals, ens_vals], axis=0)
    winners = np.argmin(stacked, axis=0)  # 0=MLP, 1=MoE, 2=ENS

    summary_rows.append({
        'metric': metric_name,
        'MLP': mlp_vals.mean(),
        'MoE': moe_vals.mean(),
        'ENS': ens_vals.mean(),
        'MLP_worst': mlp_vals.max(),
        'MoE_worst': moe_vals.max(),
        'ENS_worst': ens_vals.max(),
        # 'MLP_win_frac': (winners == 0).mean(),
        # 'MoE_win_frac': (winners == 1).mean(),
        # 'ENS_win_frac': (winners == 2).mean(),
        # 'ENS_better_than_MLP_frac': (ens_vals < mlp_vals).mean(),
        # 'ENS_better_than_MoE_frac': (ens_vals < moe_vals).mean(),
    })

inst_df = pd.DataFrame(summary_rows)
inst_df

,metric,MLP,MoE,ENS,MLP_worst,MoE_worst,ENS_worst
0,opt_gap_pct,-6.640592e+00,2.989606e+01,-2.082623e+00,18.533682,65.443629,39.797685
1,eq_violation_l1,5.276311e-05,3.854152e-05,1.768359e-05,0.001243,0.001648,0.000122
2,ineq_violation_l1,2.368882e-07,1.321528e-07,2.404113e-08,0.000045,0.000020,0.000002
3,merit_l1,2.151539e+01,2.640701e+01,1.878004e+01,142.836311,190.383344,30.171912
4,eq_violation_rate,1.369208e+00,4.417093e-01,0.000000e+00,2.461207,0.835938,0.000000
5,ineq_violation_rate,1.602909e-03,5.859375e-04,0.000000e+00,0.003906,0.001563,0.000000
6,eq_violation_l2,9.252682e-06,6.797755e-06,3.116392e-06,0.000222,0.000285,0.000021
7,ineq_violation_l2,2.242587e-07,1.242392e-07,2.342960e-08,0.000032,0.000020,0.000002
8,merit_l2,1.716308e+01,2.323184e+01,1.732326e+01,40.172052,54.106582,22.919247


In [10]:
# Optional: save tables
# out_dir = 'results/moe_vs_mlp_compare'
# os.makedirs(out_dir, exist_ok=True)

# cmp_csv = os.path.join(out_dir, 'metrics_compare.csv')
# inst_csv = os.path.join(out_dir, 'per_instance_compare.csv')
# cmp_df.to_csv(cmp_csv, index=False)
# inst_df.to_csv(inst_csv, index=False)

# print('Saved:')
# print(' -', cmp_csv)
# print(' -', inst_csv)